# Rekolte - Notebook 1b: Pre-Harvest GEE Feature Extraction

**Project:** Predicting Sugarcane Yield in Mauritius Using Satellite Imagery and Open-Source ML  
**Student:** Yashvin Booputh (M01006629)  
**Module:** CST3990 - Spring 2025/2026  

## What this notebook does

This notebook extracts a **pre-harvest feature matrix** for model v2.  
Unlike `01_ndvi_extraction.ipynb` (which used Jun-Dec in-season NDVI and produced results
only after harvest ends), this notebook extracts features **fully available before the
harvest campaign begins (April-May)**, enabling genuine pre-season TCH forecasting
for resource allocation (trucks, workers, mill capacity).

### Feature windows extracted

| Group | Window | Type | Features |
|-------|--------|------|----------|
| Lagged NDVI | Jun-Dec (Y-1) | Season-level aggregates | mean, max, std, cumulative |
| Lagged climate | Jun-Dec (Y-1) | Season-level aggregates | rainfall total, temp mean |
| Current NDVI | Oct(Y-1)-May(Y) | 8 monthly values | ndvi_oct ... ndvi_may |
| Current rainfall | Oct(Y-1)-May(Y) | 8 monthly values | rainfall_oct ... rainfall_may |
| Current temperature | Oct(Y-1)-May(Y) | 8 monthly values | temp_oct ... temp_may |

**Total: 35 features x 85 rows (seasons 2009-2025, 5 regions)**  
**Output:** `pre_harvest_features.csv`

### Design rationale
Sugarcane in Mauritius is harvested on a staggered schedule (Jun-Jan), with ratoon regrowth
starting immediately after each field is cut. Growth stages therefore overlap across fields
at all times - there is no single island-wide phenological calendar. Rather than imposing
a biologically-justified window, this design captures a rich temporal feature set across
pre-harvest months and relies on RF/XGBoost feature importance to empirically identify
the strongest predictive signals.

### Satellite strategy (same as v1)
- 2008-2012: Landsat 7 ETM+ (Collection 2 L2)
- 2013-2016: Landsat 8 OLI (Collection 2 L2)
- 2017-2025: Sentinel-2 SR Harmonised

The Oct(Y-1)-May(Y) window may straddle a satellite transition (e.g. Y=2017:
Oct-Dec 2016 uses Landsat 8, Jan-May 2017 uses Sentinel-2). These sub-windows
are extracted separately using the correct sensor and merged. No sensor dummy
features are included - NDVI normalises inter-sensor variability sufficiently
at this temporal aggregation level.

## Cell 1 - Install packages
Run once. Restart runtime if prompted.

In [1]:
!pip install earthengine-api --quiet
print('Packages ready.')

Packages ready.


## Cell 2 - Imports

In [2]:
import ee
import pandas as pd
import numpy as np
import calendar
import os
import time

print('Imports OK.')

Imports OK.


## Cell 3 - Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/CST3990 - Final Year Project'
OUTPUT_DIR = f'{DRIVE_BASE}/model_v3'
OUTPUT_CSV = f'{OUTPUT_DIR}/pre_harvest_features.csv'
CHECKPOINT = f'{OUTPUT_DIR}/pre_harvest_checkpoint.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')
print(f'Output CSV:       {OUTPUT_CSV}')

Mounted at /content/drive
Output directory: /content/drive/MyDrive/CST3990 - Final Year Project/model_v3
Output CSV:       /content/drive/MyDrive/CST3990 - Final Year Project/model_v3/pre_harvest_features.csv


## Cell 4 - Authenticate & Initialise Google Earth Engine
Follow the browser link, copy the verification code, paste it below.

In [4]:
ee.Authenticate()
ee.Initialize(project='rekolte-491422')
print('GEE initialised successfully.')

GEE initialised successfully.


## Cell 5 - Load Region Assets from GEE Cloud
Each region boundary is a MultiPolygon of individual sugarcane fields,
stored as a GEE FeatureCollection cloud asset.

In [5]:
ASSET_BASE   = 'projects/rekolte-491422/assets'
REGION_NAMES = ['NORD', 'SUD', 'EST', 'OUEST', 'CENTRE']

region_geometries = {}
for name in REGION_NAMES:
    fc = ee.FeatureCollection(f'{ASSET_BASE}/{name}_boundary')
    region_geometries[name] = fc.geometry()
    print(f'Loaded {name}_boundary')

print('\nAll 5 region assets loaded.')

Loaded NORD_boundary
Loaded SUD_boundary
Loaded EST_boundary
Loaded OUEST_boundary
Loaded CENTRE_boundary

All 5 region assets loaded.


## Cell 6 - Satellite Preprocessing Functions

### Cloud masking
- **Landsat 7/8 C2L2:** QA_PIXEL bits 3 (cloud shadow), 4 (snow/ice), 5 (cloud)
- **Sentinel-2 SR Harmonised:** QA60 bits 10 (opaque cloud), 11 (cirrus)

### NDVI
- Landsat 7: (SR_B4 - SR_B3) / (SR_B4 + SR_B3), scale=30m
- Landsat 8: (SR_B5 - SR_B4) / (SR_B5 + SR_B4), scale=30m
- Sentinel-2: (B8 - B4) / (B8 + B4), scale=10m

In [6]:
def apply_scale_factors_landsat(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True)

def mask_clouds_landsat(image):
    qa = image.select('QA_PIXEL')
    cloud        = qa.bitwiseAnd(1 << 5).eq(0)
    cloud_shadow = qa.bitwiseAnd(1 << 3).eq(0)
    snow         = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud.And(cloud_shadow).And(snow))

def preprocess_landsat7(image):
    img  = mask_clouds_landsat(apply_scale_factors_landsat(image))
    ndvi = img.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
    return img.addBands(ndvi)

def preprocess_landsat8(image):
    img  = mask_clouds_landsat(apply_scale_factors_landsat(image))
    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return img.addBands(ndvi)

def mask_clouds_sentinel2(image):
    qa = image.select('QA60')
    return image.updateMask(
        qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    )

def preprocess_sentinel2(image):
    img  = mask_clouds_sentinel2(image)
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

COLLECTIONS = {
    'landsat7':  'LANDSAT/LE07/C02/T1_L2',
    'landsat8':  'LANDSAT/LC08/C02/T1_L2',
    'sentinel2': 'COPERNICUS/S2_SR_HARMONIZED'
}
PREPROCESSORS = {
    'landsat7':  preprocess_landsat7,
    'landsat8':  preprocess_landsat8,
    'sentinel2': preprocess_sentinel2
}
SCALES = {'landsat7': 30, 'landsat8': 30, 'sentinel2': 10}

print('Preprocessing functions defined.')

Preprocessing functions defined.


## Cell 7 - Satellite Mapping & Season Config

We predict seasons **2009-2025** (17 seasons x 5 regions = 85 rows).  
`SEASON_SATELLITE` is used only for the **lagged Jun-Dec Y-1 window** (dry season,
optical satellites work reliably here).

The **current pre-harvest window (Oct Y-1 - May Y)** uses **MODIS MOD13Q1**
16-day composites for all years — see Cell 10.

In [7]:
# Harvest-season satellite — used only for lagged Jun-Dec Y-1 window
SEASON_SATELLITE = {}
for y in range(2008, 2013): SEASON_SATELLITE[y] = 'landsat7'
for y in range(2013, 2017): SEASON_SATELLITE[y] = 'landsat8'
for y in range(2017, 2026): SEASON_SATELLITE[y] = 'sentinel2'

PREDICTION_YEARS = list(range(2009, 2026))

print(f'Prediction years: {PREDICTION_YEARS[0]}-{PREDICTION_YEARS[-1]}')
print(f'Total combos: {len(PREDICTION_YEARS)} x {len(REGION_NAMES)} = '
      f'{len(PREDICTION_YEARS) * len(REGION_NAMES)} rows')
print('Lagged window (Jun-Dec Y-1): Landsat 7 / 8 / Sentinel-2')
print('Current window (Oct Y-1 - May Y): MODIS MOD13Q1 (all years)')

Prediction years: 2009-2025
Total combos: 17 x 5 = 85 rows
Lagged window (Jun-Dec Y-1): Landsat 7 / 8 / Sentinel-2
Current window (Oct Y-1 - May Y): MODIS MOD13Q1 (all years)


## Cell 8 - Core NDVI Windowed Extractor

`get_ndvi_df()` fetches all cloud-free NDVI observations in a date window and
returns a DataFrame of `(date, year, month, ndvi)`.  
Used by both the lagged and current window extractors.

Each image is reduced to its **mean NDVI over the region geometry**,
producing one row per cloud-free overpass.

In [8]:
def get_ndvi_df(start_year, start_month, end_year, end_month, satellite, region_geom):
    """
    Return DataFrame of cloud-free NDVI observations for a given window + satellite.
    Columns: date (Timestamp), year (int), month (int), ndvi (float)
    Returns empty DataFrame if no valid imagery found.
    """
    scale = SCALES[satellite]
    _, last_day = calendar.monthrange(end_year, end_month)
    start_str = f'{start_year}-{start_month:02d}-01'
    end_str   = f'{end_year}-{end_month:02d}-{last_day:02d}'

    col = (ee.ImageCollection(COLLECTIONS[satellite])
           .filterDate(start_str, end_str)
           .filterBounds(region_geom)
           .map(PREPROCESSORS[satellite])
           .select('NDVI'))

    if col.size().getInfo() == 0:
        return pd.DataFrame(columns=['date', 'year', 'month', 'ndvi'])

    def img_to_feat(img):
        val = img.reduceRegion(
            reducer    = ee.Reducer.mean(),
            geometry   = region_geom,
            scale      = scale,
            maxPixels  = 1e10,
            bestEffort = True
        ).get('NDVI')
        return ee.Feature(None, {'ts': img.date().millis(), 'ndvi': val})

    rows = ee.FeatureCollection(col.map(img_to_feat)).getInfo()['features']
    ts   = [r['properties'].get('ts')   for r in rows]
    vals = [r['properties'].get('ndvi') for r in rows]

    df = pd.DataFrame({'date': pd.to_datetime(ts, unit='ms'), 'ndvi': vals})
    df = df.dropna(subset=['ndvi'])
    df['year']  = df['date'].dt.year
    df['month'] = df['date'].dt.month
    return df

print('get_ndvi_df() defined.')

get_ndvi_df() defined.


## Cell 9 - Lagged NDVI Aggregates (Jun-Dec Y-1)

Captures the **previous harvest season's crop vigour** as 4 season-level statistics:
- `ndvi_lag_mean` - mean of 7 monthly means (Jun-Dec)
- `ndvi_lag_max` - peak monthly mean (best single-month canopy density)
- `ndvi_lag_std` - standard deviation of monthly means (intra-season variability)
- `ndvi_lag_cumulative` - sum of monthly means (proxy for total green biomass over season)

**Why aggregate to monthly means first?**  
Computing season stats directly on all observations would introduce observation-count bias:
years with more cloud-free images would artificially inflate cumulative/mean values.
Aggregating to monthly means first removes this dependency.

In [9]:
def extract_lagged_ndvi(pred_year, region_geom):
    """
    Extract Jun-Dec(pred_year-1) NDVI as 4 season-level aggregates.

    Returns dict: ndvi_lag_mean, ndvi_lag_max, ndvi_lag_std,
                  ndvi_lag_cumulative, obs_count_lag
    """
    lag_year  = pred_year - 1
    satellite = SEASON_SATELLITE[lag_year]

    df = get_ndvi_df(lag_year, 6, lag_year, 12, satellite, region_geom)

    empty = {
        'ndvi_lag_mean': None, 'ndvi_lag_max': None,
        'ndvi_lag_std':  None, 'ndvi_lag_cumulative': None,
        'obs_count_lag': 0
    }
    if len(df) == 0:
        return empty

    monthly  = df.groupby('month')['ndvi'].mean()
    n_months = len(monthly)

    return {
        'ndvi_lag_mean':       round(float(monthly.mean()), 6),
        'ndvi_lag_max':        round(float(monthly.max()),  6),
        'ndvi_lag_std':        round(float(monthly.std()),  6) if n_months > 1 else 0.0,
        'ndvi_lag_cumulative': round(float(monthly.sum()),  6),
        'obs_count_lag':       len(df)
    }

print('extract_lagged_ndvi() defined.')

extract_lagged_ndvi() defined.


## Cell 10 - Current NDVI Monthly (Oct Y-1 to May Y) — MODIS MOD13Q1

The pre-harvest window (Oct-May) overlaps with Mauritius's wet and cyclone season
(Nov-April). Single-date optical sensors (Landsat, Sentinel-2) are blocked by
persistent cloud cover, returning 0 valid observations for many years.

**Solution: MODIS MOD13Q1 16-day composites**  
MODIS takes daily imagery and selects the clearest pixel across each 16-day window,
making it cloud-robust and reliable year-round. One consistent sensor across all
17 seasons — no satellite transitions, no mixing.

| Property | Value |
|----------|-------|
| Collection | `MODIS/061/MOD13Q1` |
| Resolution | 250m |
| Cadence | 16-day composite |
| NDVI scale | x 0.0001 |
| Quality filter | `SummaryQA <= 1` (good + marginal; excludes snow and cloudy) |
| Available | 2000-02-18 to present |

At 250m over large regional polygons (tens of km2), the coarser resolution
has negligible impact on the regional mean NDVI used as model input.

In [10]:
def get_modis_ndvi_df(lag_year, pred_year, region_geom):
    """
    Extract MODIS MOD13Q1 16-day NDVI composites for Oct(lag_year)-May(pred_year).

    Quality filter: SummaryQA <= 1 (good + marginal pixels only)
    NDVI scale:     multiply by 0.0001
    Resolution:     250m

    Returns DataFrame: date, year, month, ndvi
    """
    start_str = f'{lag_year}-10-01'
    end_str   = f'{pred_year}-06-01'  # exclusive end — captures all of May

    col = (ee.ImageCollection('MODIS/061/MOD13Q1')
           .filterDate(start_str, end_str)
           .filterBounds(region_geom)
           .select(['NDVI', 'SummaryQA']))

    if col.size().getInfo() == 0:
        return pd.DataFrame(columns=['date', 'year', 'month', 'ndvi'])

    def mask_and_scale(img):
        """Apply QA mask and scale NDVI to -1..1 range."""
        qa   = img.select('SummaryQA')
        good = qa.lte(1)  # 0=good, 1=marginal; excludes 2=snow, 3=cloudy
        ndvi = img.select('NDVI').multiply(0.0001).updateMask(good)
        return ndvi.rename('NDVI').copyProperties(img, ['system:time_start'])

    col_clean = col.map(mask_and_scale)

    def img_to_feat(img):
        val = img.reduceRegion(
            reducer    = ee.Reducer.mean(),
            geometry   = region_geom,
            scale      = 250,
            maxPixels  = 1e9,
            bestEffort = True
        ).get('NDVI')
        return ee.Feature(None, {'ts': img.date().millis(), 'ndvi': val})

    rows = ee.FeatureCollection(col_clean.map(img_to_feat)).getInfo()['features']
    ts   = [r['properties'].get('ts')   for r in rows]
    vals = [r['properties'].get('ndvi') for r in rows]

    df = pd.DataFrame({'date': pd.to_datetime(ts, unit='ms'), 'ndvi': vals})
    df = df.dropna(subset=['ndvi'])
    df['year']  = df['date'].dt.year
    df['month'] = df['date'].dt.month
    return df


def extract_current_ndvi_monthly(pred_year, region_geom):
    """
    Extract 8 monthly NDVI means for Oct(Y-1)-May(Y) using MODIS MOD13Q1.
    Single consistent sensor across all prediction years 2009-2025.

    Returns dict: ndvi_oct, ndvi_nov, ndvi_dec (Y-1),
                  ndvi_jan, ndvi_feb, ndvi_mar, ndvi_apr, ndvi_may (Y),
                  obs_count_oct_dec, obs_count_jan_may
    """
    lag_year = pred_year - 1

    df = get_modis_ndvi_df(lag_year, pred_year, region_geom)

    # month -> (column key, which year it belongs to)
    MONTH_CFG = [
        (10, 'ndvi_oct', lag_year),  (11, 'ndvi_nov', lag_year),
        (12, 'ndvi_dec', lag_year),
        (1,  'ndvi_jan', pred_year), (2,  'ndvi_feb', pred_year),
        (3,  'ndvi_mar', pred_year), (4,  'ndvi_apr', pred_year),
        (5,  'ndvi_may', pred_year)
    ]

    result = {key: None for _, key, _ in MONTH_CFG}

    for month_num, key, year in MONTH_CFG:
        vals = df.loc[(df['month'] == month_num) & (df['year'] == year), 'ndvi']
        result[key] = round(float(vals.mean()), 6) if len(vals) > 0 else None

    oct_dec = df[df['month'].isin([10, 11, 12])]
    jan_may = df[df['month'].isin([1, 2, 3, 4, 5])]
    result['obs_count_oct_dec'] = len(oct_dec)
    result['obs_count_jan_may'] = len(jan_may)
    return result

print('MODIS extractor defined.')
print('Collection: MODIS/061/MOD13Q1 | Resolution: 250m | QA: SummaryQA <= 1')

MODIS extractor defined.
Collection: MODIS/061/MOD13Q1 | Resolution: 250m | QA: SummaryQA <= 1


## Cell 11 - Climate Extraction Helpers

- `get_chirps_monthly(year, month)` - sums CHIRPS daily precipitation to monthly total (mm)
- `get_era5_monthly(year, month)` - returns ERA5-Land monthly mean 2m temperature (deg C)

Both return `None` if the collection is empty for the requested period.

In [11]:
def get_chirps_monthly(year, month, region_geom):
    """CHIRPS daily precip summed to monthly total (mm). Returns None if no data."""
    _, last_day = calendar.monthrange(year, month)
    col = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
           .filterDate(f'{year}-{month:02d}-01',
                       f'{year}-{month:02d}-{last_day:02d}')
           .select('precipitation'))
    if col.size().getInfo() == 0:
        return None
    result = col.sum().reduceRegion(
        reducer    = ee.Reducer.mean(),
        geometry   = region_geom,
        scale      = 5566,
        maxPixels  = 1e9,
        bestEffort = True
    ).getInfo()
    val = result.get('precipitation')
    return round(val, 2) if val is not None else None


def get_era5_monthly(year, month, region_geom):
    """ERA5-Land monthly mean 2m temperature in deg C. Returns None if no data."""
    next_month = month + 1 if month < 12 else 1
    next_year  = year if month < 12 else year + 1
    col = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
           .filterDate(f'{year}-{month:02d}-01',
                       f'{next_year}-{next_month:02d}-01')
           .select('temperature_2m'))
    if col.size().getInfo() == 0:
        return None
    result = col.mean().reduceRegion(
        reducer    = ee.Reducer.mean(),
        geometry   = region_geom,
        scale      = 11132,
        maxPixels  = 1e9,
        bestEffort = True
    ).getInfo()
    temp_k = result.get('temperature_2m')
    return round(temp_k - 273.15, 3) if temp_k is not None else None

print('Climate helpers defined.')

Climate helpers defined.


## Cell 12 - Lagged Climate Aggregates (Jun-Dec Y-1)

Extracts 2 season-level aggregates from the previous harvest season:
- `rainfall_lag_total` - total CHIRPS precipitation (mm) over Jun-Dec(Y-1)
- `temp_lag_mean` - mean ERA5 temperature (deg C) over Jun-Dec(Y-1)

In [12]:
def extract_lagged_climate(pred_year, region_geom):
    """
    Extract Jun-Dec(pred_year-1) climate as 2 season-level aggregates.
    Returns dict: rainfall_lag_total, temp_lag_mean
    """
    lag_year  = pred_year - 1
    rain_vals = []
    temp_vals = []

    for month in range(6, 13):
        r = get_chirps_monthly(lag_year, month, region_geom)
        t = get_era5_monthly(lag_year, month, region_geom)
        if r is not None:
            rain_vals.append(r)
        if t is not None:
            temp_vals.append(t)

    return {
        'rainfall_lag_total': round(sum(rain_vals), 1)                  if rain_vals else None,
        'temp_lag_mean':      round(sum(temp_vals) / len(temp_vals), 3) if temp_vals else None
    }

print('extract_lagged_climate() defined.')

extract_lagged_climate() defined.


## Cell 13 - Current Climate Monthly (Oct Y-1 to May Y)

Extracts 8 monthly CHIRPS and ERA5 values across the pre-harvest window.  
Oct-Dec values use the lag year, Jan-May values use the prediction year.

In [13]:
# (year_flag, month_number, column_suffix)
# 'lag' = pred_year-1, 'cur' = pred_year
CURRENT_MONTHS_CFG = [
    ('lag', 10, 'oct'), ('lag', 11, 'nov'), ('lag', 12, 'dec'),
    ('cur',  1, 'jan'), ('cur',  2, 'feb'), ('cur',  3, 'mar'),
    ('cur',  4, 'apr'), ('cur',  5, 'may'),
]


def extract_current_climate_monthly(pred_year, region_geom):
    """
    Extract 8 monthly CHIRPS + ERA5 values for Oct(Y-1)-May(Y).
    Returns dict with 16 keys: rainfall_oct..rainfall_may, temp_oct..temp_may
    """
    lag_year = pred_year - 1
    result   = {}
    for yr_flag, month, key in CURRENT_MONTHS_CFG:
        year = lag_year if yr_flag == 'lag' else pred_year
        result[f'rainfall_{key}'] = get_chirps_monthly(year, month, region_geom)
        result[f'temp_{key}']     = get_era5_monthly(year, month, region_geom)
    return result

print('extract_current_climate_monthly() defined.')

extract_current_climate_monthly() defined.


## Cell 14 - Main Extraction Loop

Iterates over all 85 combinations (17 seasons x 5 regions).  
For each combination, 4 extraction functions are called in sequence:
1. `extract_lagged_ndvi` - 1 GEE collection query (Jun-Dec Y-1)
2. `extract_lagged_climate` - 14 GEE calls (7 months x CHIRPS + ERA5)
3. `extract_current_ndvi_monthly` - 2 GEE collection queries (Oct-Dec + Jan-May)
4. `extract_current_climate_monthly` - 16 GEE calls (8 months x CHIRPS + ERA5)

**Estimated runtime:** 60-90 minutes depending on GEE server load.  
Checkpoint saved every 5 records to recover from interruptions.

In [14]:
total      = len(PREDICTION_YEARS) * len(REGION_NAMES)
results    = []
done       = 0
start_time = time.time()

# Column list for null records on error
ALL_COLS = [
    'ndvi_lag_mean', 'ndvi_lag_max', 'ndvi_lag_std', 'ndvi_lag_cumulative', 'obs_count_lag',
    'rainfall_lag_total', 'temp_lag_mean',
    'ndvi_oct', 'ndvi_nov', 'ndvi_dec',
    'ndvi_jan', 'ndvi_feb', 'ndvi_mar', 'ndvi_apr', 'ndvi_may',
    'rainfall_oct', 'rainfall_nov', 'rainfall_dec',
    'rainfall_jan', 'rainfall_feb', 'rainfall_mar', 'rainfall_apr', 'rainfall_may',
    'temp_oct', 'temp_nov', 'temp_dec',
    'temp_jan', 'temp_feb', 'temp_mar', 'temp_apr', 'temp_may',
    'obs_count_oct_dec', 'obs_count_jan_may'
]

print(f'Pre-harvest feature extraction: {total} combinations')
print(f'Seasons {PREDICTION_YEARS[0]}-{PREDICTION_YEARS[-1]} | Regions: {REGION_NAMES}\n')

for pred_year in PREDICTION_YEARS:
    for region in REGION_NAMES:
        done   += 1
        elapsed = time.time() - start_time
        eta     = (elapsed / done) * (total - done) if done > 1 else 0
        print(f'[{done:2d}/{total}] {pred_year} {region:<7} ', end='', flush=True)

        try:
            geom = region_geometries[region]
            rec  = {'season': pred_year, 'region': region}

            # 1. Lagged NDVI aggregates (Jun-Dec Y-1)
            rec.update(extract_lagged_ndvi(pred_year, geom))

            # 2. Lagged climate aggregates (Jun-Dec Y-1)
            rec.update(extract_lagged_climate(pred_year, geom))

            # 3. Current NDVI monthly (Oct Y-1 - May Y)
            rec.update(extract_current_ndvi_monthly(pred_year, geom))

            # 4. Current climate monthly (Oct Y-1 - May Y)
            rec.update(extract_current_climate_monthly(pred_year, geom))

            results.append(rec)

            lag_str = f"lag={rec['ndvi_lag_mean']:.3f}" if rec.get('ndvi_lag_mean') is not None else 'lag=None'
            may_str = f"may={rec['ndvi_may']:.3f}"     if rec.get('ndvi_may')      is not None else 'may=None'
            print(f'{lag_str}  {may_str}  ETA={eta/60:.1f}min')

        except Exception as e:
            import traceback
            print(f'ERROR: {e}')
            traceback.print_exc()
            rec = {'season': pred_year, 'region': region}
            for col in ALL_COLS:
                rec[col] = None
            results.append(rec)

        if done % 5 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT, index=False)
            print(f'  -> checkpoint saved ({done}/{total}, ETA {eta/60:.1f} min)')

df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_CSV, index=False)
elapsed_total = (time.time() - start_time) / 60
print(f'\nDone in {elapsed_total:.1f} min')
print(f'Saved {len(df_out)} records -> {OUTPUT_CSV}')

Pre-harvest feature extraction: 85 combinations
Seasons 2009-2025 | Regions: ['NORD', 'SUD', 'EST', 'OUEST', 'CENTRE']

[ 1/85] 2009 NORD    lag=0.446  may=0.797  ETA=0.0min
[ 2/85] 2009 SUD     lag=0.456  may=0.791  ETA=9.5min
[ 3/85] 2009 EST     lag=0.470  may=0.786  ETA=15.2min
[ 4/85] 2009 OUEST   lag=0.395  may=0.772  ETA=17.2min
[ 5/85] 2009 CENTRE  lag=0.504  may=0.735  ETA=16.1min
  -> checkpoint saved (5/85, ETA 16.1 min)
[ 6/85] 2010 NORD    lag=0.551  may=0.785  ETA=15.1min
[ 7/85] 2010 SUD     lag=0.516  may=0.778  ETA=16.0min
[ 8/85] 2010 EST     lag=0.520  may=0.781  ETA=16.6min
[ 9/85] 2010 OUEST   lag=0.454  may=0.743  ETA=15.7min
[10/85] 2010 CENTRE  lag=0.539  may=0.743  ETA=15.2min
  -> checkpoint saved (10/85, ETA 15.2 min)
[11/85] 2011 NORD    lag=0.398  may=0.807  ETA=15.0min
[12/85] 2011 SUD     lag=0.390  may=0.787  ETA=15.1min
[13/85] 2011 EST     lag=0.403  may=0.773  ETA=15.7min
[14/85] 2011 OUEST   lag=0.345  may=0.788  ETA=15.2min
[15/85] 2011 CENTRE  lag=

## Cell 15 - Validation & QA

Verify shape, missing values, and feature distributions before handing off to
`02_pre_harvest_training.ipynb`.  
Some missing monthly NDVI values are expected (persistent cloud cover in certain
months/regions) and will be imputed in the training notebook.

In [15]:
df_check = pd.read_csv(OUTPUT_CSV)

print('=== Shape ===')
print(f'{df_check.shape[0]} rows x {df_check.shape[1]} columns')
print(f'Expected: 85 rows x 38 columns (35 features + season + region + 3 QA obs counts)')

print('\n=== Missing values per column ===')
missing = df_check.isnull().sum()
missing = missing[missing > 0]
if len(missing):
    print(missing.to_string())
    print('\nNote: Missing NDVI will be imputed with row mean in training notebook.')
else:
    print('No missing values.')

print('\n=== Seasons & Regions ===')
print(f'Seasons: {sorted(df_check["season"].unique())}')
print(f'Regions: {sorted(df_check["region"].unique())}')

print('\n=== Observation counts (median per season) ===')
obs_cols = [c for c in df_check.columns if 'obs_count' in c]
print(df_check.groupby('season')[obs_cols].median().to_string())

print('\n=== NDVI feature means by region ===')
ndvi_cols = [c for c in df_check.columns
             if c.startswith('ndvi_') and 'count' not in c]
print(df_check.groupby('region')[ndvi_cols].mean().round(3).to_string())

print('\n=== First 5 rows ===')
df_check.head()

=== Shape ===
85 rows x 35 columns
Expected: 85 rows x 38 columns (35 features + season + region + 3 QA obs counts)

=== Missing values per column ===
No missing values.

=== Seasons & Regions ===
Seasons: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Regions: ['CENTRE', 'EST', 'NORD', 'OUEST', 'SUD']

=== Observation counts (median per season) ===
        obs_count_lag  obs_count_oct_dec  obs_count_jan_may
season                                                     
2009              5.0                5.0               10.0
2010              6.0                5.0               10.0
2011              2.0                5.0               10.0
2012              9.0                5.0               10.0
2013             11.0                5.0               10.0
20

,season,region,ndvi_lag_mean,ndvi_lag_max,ndvi_lag_std,ndvi_lag_cumulative,obs_count_lag,rainfall_lag_total,temp_lag_mean,ndvi_oct,...,rainfall_jan,temp_jan,rainfall_feb,temp_feb,rainfall_mar,temp_mar,rainfall_apr,temp_apr,rainfall_may,temp_may
0,2009,NORD,0.446375,0.459928,0.014012,1.339124,5,640.6,22.470,0.513352,...,173.80,26.690,232.35,26.161,369.87,25.764,203.68,25.109,148.34,23.605
1,2009,SUD,0.455916,0.497609,0.037845,1.367749,5,863.7,22.422,0.499347,...,204.23,26.698,229.48,26.125,293.66,25.760,280.14,25.052,167.94,23.476
2,2009,EST,0.469830,0.521371,0.044756,1.409489,5,1014.8,22.335,0.486945,...,230.62,26.563,274.03,26.038,464.64,25.632,355.20,24.986,183.36,23.435
3,2009,OUEST,0.394529,0.410744,0.015612,1.183588,5,856.4,21.292,0.490202,...,206.03,25.753,286.53,25.224,347.72,24.971,235.00,24.181,174.32,22.438
4,2009,CENTRE,0.504154,0.528103,0.021684,1.512463,5,895.7,21.641,0.537440,...,255.00,26.002,292.51,25.452,414.83,25.090,306.99,24.392,185.88,22.750


## Cell 16 - MODIS Patch: Replace Current-Window NDVI for All 85 Rows

Run this cell if you already have a `pre_harvest_features.csv` from a previous
run that used Landsat/Sentinel-2 for the Oct-May window (which had cloud gaps).

This cell re-extracts **only the 8 monthly NDVI columns** (ndvi_oct...ndvi_may)
using MODIS for all 85 rows, and patches them into the existing CSV.
The lagged NDVI + all climate features are preserved — no need to re-extract those.

**Estimated runtime:** ~15-20 minutes (MODIS: one query per row, much faster than
individual Landsat/S2 monthly calls).

In [16]:
import pandas as pd
import time

# Columns to replace (MODIS will provide these)
MODIS_COLS = ['ndvi_oct', 'ndvi_nov', 'ndvi_dec',
              'ndvi_jan', 'ndvi_feb', 'ndvi_mar', 'ndvi_apr', 'ndvi_may',
              'obs_count_oct_dec', 'obs_count_jan_may']

# Load existing CSV (lagged NDVI + climate already extracted)
df_existing = pd.read_csv(OUTPUT_CSV)
print(f'Loaded existing CSV: {df_existing.shape}')
print(f'Patching {len(MODIS_COLS)} columns with MODIS values for all 85 rows...')
print(f'Preserving: lagged NDVI, rainfall, temperature columns\n')

total_p    = len(PREDICTION_YEARS) * len(REGION_NAMES)
patch_rows = []
done_p     = 0
start_p    = time.time()

for pred_year in PREDICTION_YEARS:
    for region in REGION_NAMES:
        done_p += 1
        elapsed = time.time() - start_p
        eta     = (elapsed / done_p) * (total_p - done_p) if done_p > 1 else 0
        print(f'[{done_p:2d}/{total_p}] {pred_year} {region:<7} ', end='', flush=True)

        try:
            modis_rec = extract_current_ndvi_monthly(
                pred_year, region_geometries[region]
            )
            modis_rec['season'] = pred_year
            modis_rec['region'] = region
            patch_rows.append(modis_rec)

            jan = modis_rec.get('ndvi_jan')
            may = modis_rec.get('ndvi_may')
            obs = modis_rec.get('obs_count_jan_may', 0)
            jan_str = f'jan={jan:.3f}' if jan is not None else 'jan=None'
            may_str = f'may={may:.3f}' if may is not None else 'may=None'
            print(f'{jan_str}  {may_str}  obs={obs}  ETA={eta/60:.1f}min')

        except Exception as e:
            import traceback
            print(f'ERROR: {e}')
            traceback.print_exc()
            patch_rows.append({
                'season': pred_year, 'region': region,
                **{c: None for c in MODIS_COLS}
            })

# Merge patch into existing DataFrame
df_patch = pd.DataFrame(patch_rows)[['season', 'region'] + MODIS_COLS]

# Drop old MODIS columns from existing CSV and replace with new ones
cols_to_drop = [c for c in MODIS_COLS if c in df_existing.columns]
df_existing  = df_existing.drop(columns=cols_to_drop)
df_updated   = df_existing.merge(df_patch, on=['season', 'region'], how='left')
df_updated   = df_updated.sort_values(['season', 'region']).reset_index(drop=True)
df_updated.to_csv(OUTPUT_CSV, index=False)

elapsed_total = (time.time() - start_p) / 60
print(f'\nDone in {elapsed_total:.1f} min — saved {len(df_updated)} rows')

# Quick QA
df_qa = pd.read_csv(OUTPUT_CSV)
print(f'\n=== Post-patch QA ===')
print(f'Shape: {df_qa.shape}')
monthly_cols = ['ndvi_oct','ndvi_nov','ndvi_dec','ndvi_jan','ndvi_feb',
                'ndvi_mar','ndvi_apr','ndvi_may']
missing = df_qa[monthly_cols].isnull().sum()
print('Missing monthly NDVI:')
print(missing[missing > 0].to_string() if missing.any() else '  None — all complete!')
print('\nObservation counts (median per season):')
print(df_qa.groupby('season')[['obs_count_oct_dec','obs_count_jan_may']].median().to_string())

Loaded existing CSV: (85, 35)
Patching 10 columns with MODIS values for all 85 rows...
Preserving: lagged NDVI, rainfall, temperature columns

[ 1/85] 2009 NORD    jan=0.645  may=0.797  obs=10  ETA=0.0min
[ 2/85] 2009 SUD     jan=0.649  may=0.791  obs=10  ETA=0.3min
[ 3/85] 2009 EST     jan=0.664  may=0.786  obs=10  ETA=0.5min
[ 4/85] 2009 OUEST   jan=0.654  may=0.772  obs=10  ETA=0.5min
[ 5/85] 2009 CENTRE  jan=0.651  may=0.735  obs=10  ETA=0.4min
[ 6/85] 2010 NORD    jan=0.676  may=0.785  obs=10  ETA=0.4min
[ 7/85] 2010 SUD     jan=0.664  may=0.778  obs=10  ETA=0.4min
[ 8/85] 2010 EST     jan=0.646  may=0.781  obs=10  ETA=0.5min
[ 9/85] 2010 OUEST   jan=0.737  may=0.743  obs=10  ETA=0.5min
[10/85] 2010 CENTRE  jan=0.694  may=0.743  obs=10  ETA=0.5min
[11/85] 2011 NORD    jan=0.538  may=0.807  obs=10  ETA=0.5min
[12/85] 2011 SUD     jan=0.540  may=0.787  obs=10  ETA=0.7min
[13/85] 2011 EST     jan=0.505  may=0.773  obs=10  ETA=0.6min
[14/85] 2011 OUEST   jan=0.550  may=0.788  obs=10  